원본 파일 : utils/namuwili_extract.py로 추출하고 pretraining/utils/clean_corpus.py에서 가져옴

In [1]:
import re
import os
import string
from tqdm import tqdm
from collections import Counter
import html

In [2]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CORPUS_DATA = os.path.join(REPO_ROOT, "data", "1.corpus")

# ==========================================
# 경로 및 설정
# ==========================================
INPUT_CORPUS = 'namuwiki'
DATASET_PATH = "origin/namuwiki_medical.txt"

# 출력 경로 직접 지정
OUTPUT_DIR = os.path.join(CORPUS_DATA, "1.wiki/cleaned")
OUTPUT_PATH = os.path.join(OUTPUT_DIR, "namuwiki_parsed.txt")

# 출력 폴더가 없으면 생성 (에러 방지)
os.makedirs(OUTPUT_DIR, exist_ok=True)

print(f"입력 파일: {DATASET_PATH}")
print(f"저장 경로: {OUTPUT_PATH}")

입력 파일: origin/namuwiki_medical.txt
저장 경로: ./cleaned/namuwiki_parsed.txt


In [3]:
def clean_str(text: str) -> str:
    PUNCT = '\\'.join(string.punctuation)
    # 특수문자 매핑 및 제거 로직
    PUNCT_MAPPING = {
        "‘": "'", "₹": "e", "´": "'", "°": "", "€": "e", "™": "tm", "√": " sqrt ", "×": "x", "²": "2",
        "—": "-", "–": "-", "’": "'", '”': '"', '“': '"', "£": "e", '∞': 'infinity',
        'θ': 'theta', '÷': '/', 'α': 'alpha', '•': '.', 'à': 'a', '−': '-', 'β': 'beta', '∅': '', '³': '3', 'π': 'pi',
        '\u200b': ' ', '…': ' ... ', '\ufeff': '', 'करना': '', 'है': '', '·': '.'
    }
    HTML = re.compile('<[^>]*>')
    # 화이트리스트: 영문, 숫자, 한글(자모 포함), 구두점, 공백만 남김
    ONLY_REMAIN = re.compile(rf"[^A-Za-z0-9ㄱ-ㅣ가-힣{PUNCT}\s]")

    text = re.sub(HTML, "", text)

    for p in PUNCT_MAPPING:
        text = text.replace(p, PUNCT_MAPPING[p])
    
    # 화이트리스트 이외 문자 제거
    text = re.sub(ONLY_REMAIN, "", text)

    # 다중 공백 제거
    text = re.sub(r'\s+', ' ', text).strip()

    return text

In [4]:
# 메인 로직 실행
if not os.path.exists(DATASET_PATH):
    print(f"[ERROR] 파일이 존재하지 않습니다: {DATASET_PATH}")
else:
    temp_dataset = []
    doc = []
    
    # 파일 읽기
    with open(DATASET_PATH, "r", encoding="utf-8") as fin:
        dataset = fin.readlines()
        data_size = len(dataset)

        for line in tqdm(dataset, desc='Cleaning the data...', total=data_size, bar_format="{l_bar}{bar:15}{r_bar}"):
            # Namuwiki 문서 구분 로직 (빈 줄로 구분)
            if INPUT_CORPUS == 'namuwiki' and line == '\n':
                if doc: # doc이 비어있지 않을 때만 추가
                    temp_dataset.append(doc)
                    doc = []
                continue
            
            # Kowiki 문서 구분 로직
            elif INPUT_CORPUS == 'kowiki':
                if line.isspace() or len(line) == 0:
                    continue
                elif '</doc>' in line:
                    temp_dataset.append(doc)
                    doc = []
                    continue

            line = line.strip()
            line = clean_str(line)
            
            if line.isspace() or len(line.strip()) == 0:
                continue
            
            line = " ".join(line.split())
            line = line.replace('( )', '()')
            doc.append(line)
        
        # 마지막 문서가 남아있다면 추가
        if doc:
            temp_dataset.append(doc)
    num_docs = len(temp_dataset)
    
    # 결과 저장
    with open(OUTPUT_PATH, "w", encoding="utf-8") as fout:
        for doc in tqdm(temp_dataset, desc='Saving the data...', total=num_docs, bar_format="{l_bar}{bar:15}{r_bar}"):
            if len(doc) < 2:
                continue

            for line in doc:
                if line.isspace() or len(line) == 0:
                    continue
                else:
                    fout.write(line + '\n')
            fout.write('\n\n') # 문서 간 구분

    print(f"Done. Saved to: {OUTPUT_PATH}")

Saving the data...: 100%|███████████████| 45863/45863 [00:00<00:00, 378068.19it/s]

Done. Saved to: ./cleaned/namuwiki_parsed.txt


### 위키 문법 지우개

In [5]:
# ==========================================
# 설정
# ==========================================
INPUT_FILE = "cleaned/namuwiki_parsed.txt"
OUTPUT_FILE = "cleaned/namuwiki_cleaned.txt"

# =========================================
# 🧠 스마트 스팸 판별 함수 (길이가 길어도 내용은 검사함)
# ==========================================
def is_spam_content(text):
    length = len(text)
    
    # 1. 너무 짧은 문장 (20자 미만) -> 버림
    if length < 20:
        return True

    boilerplate_keywords = [
        "CC BY-NC-SA", "저작권", "크리에이티브 커먼즈", # 라이선스
        "라이선스", "기여하신 문서의", "문서는","돌아가기" # 안내 문구
        "편집", "토론", "역사", "ACL", # 위키 UI 텍스트
        "분류:", "목차", "파일:", "틀:", "각주" # 위키 메타데이터
    ]
    
    # 문장이 짧은데(50자 미만) 위 키워드가 포함되어 있다면 스팸 처리
    if length < 50:
        for keyword in boilerplate_keywords:
            if keyword in text:
                return True

    # B. 특수문자로 시작하는 목록형 노이즈 (의미 없는 나열일 가능성)
    # 예: * 2024-01-01, > 인용구만 덩그러니 있는 경우
    if text.strip().startswith(('*', '>', '-')):
        # 내용 없이 날짜나 숫자만 있는 경우 필터링
        content_without_symbol = text.strip()[1:].strip()
        if len(content_without_symbol) < 5:
            return True

    # 길이가 짧으면(100자 미만) 더 이상 검사 안 하고 통과 (일반 문장일 확률 높음)
    if length < 300:
        return False

    # ---------------------------------------------------------
    # 여기서부터는 긴 문장(300자 이상)에 대한 정밀 검사
    # ---------------------------------------------------------

    # 2. [언어 밀도 검사] Line 7330076 (숫자 나열) 잡는 로직
    # 한글(가-힣)과 영어(a-zA-Z)가 전체 길이의 20%도 안 되면 스팸 (기계어, 숫자 덤프 등)
    # 공백을 제외한 실제 문자 길이 대비 유효 문자 비율 계산
    text_no_space = text.replace(" ", "")
    if len(text_no_space) == 0: return True
    
    valid_chars = len(re.sub(r'[^가-힣a-zA-Z]', '', text_no_space))
    if (valid_chars / len(text_no_space)) < 0.20:
        return True # 스팸 판정 (유효 문자가 너무 적음)

    # 3. [단어 도배 검사] Line 1343555 (Ook. Ook.) 잡는 로직
    # 공백 기준으로 단어를 쪼갰을 때, 특정 단어가 절반 이상을 차지하면 도배로 간주
    words = text.split()
    if not words: return True
    
    # 가장 흔한 단어 1개 추출
    most_common_word, count = Counter(words).most_common(1)[0]
    
    # 그 단어가 전체 단어 수의 60%를 넘으면 스팸으로 의심
    if (count / len(words)) > 0.6:
        return True 

    # 4. [공백 실종 검사] Base64 이미지 코드 등
    # 1000자 넘는데 공백이 1% 미만이면 사람이 쓴 글 아님
    if length > 1000 and (text.count(' ') / length) < 0.01:
        return True

    return False # 모든 검사를 통과하면 정상 문장 (Line 8261877 소설은 여기로 통과됨)


# ==========================================
# 🧹 텍스트 정제 함수
# ==========================================
def clean_text_final(text):
# 1. [HTML Entities] &mdash;, &#10; 등을 문자로 변환
    text = html.unescape(text)

    # 2. [Wiki Syntax - Math/Macro] 수식과 매크로 먼저 삭제
    text = re.sub(r'\[math\(.*?\)\]', '', text) 
    text = re.sub(r'\{\{\{.*?\}\}\}', '', text)
    text = re.sub(r'\[include\(.*?\)\]', '', text, flags=re.IGNORECASE)

    # 3. [Wiki Syntax - Link] [[Target|Label]] -> Label 추출
    text = re.sub(r'\[\[(?:[^|\]]*\|)?([^|\]]+)\]\]', r'\1', text)
    
    # 4. 남은 괄호 껍데기 제거
    text = re.sub(r'[\[\]\{\}]', '', text)

    # 5. [URL] http://... 제거
    text = re.sub(r'(https?://|www\.)[a-zA-Z0-9가-힣./?=&%~_#-]+', '', text)

    # 6. [Email] 이메일 제거
    text = re.sub(r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}', '', text)

    # 7. [수정됨] 마침표 정제 로직
    # (1) 괄호 안에 마침표만 1개 이상 있는 경우 ((.), (...) 등) -> 삭제 (무의미한 노이즈)
    text = re.sub(r'\(\.+\)', '', text)
    
    # (2) 마침표가 2개 이상 반복되는 경우 -> 마침표 하나(.)로 치환 (문장 경계 보존)
    text = re.sub(r'\.{2,}', '.', text)

    # 8. [특수문자 청소]
    text = re.sub(r'[^가-힣a-zA-Z0-9\s.,?!\'"~%\(\)-]+', ' ', text)
                     
    # 9. 공백 정리
    text = re.sub(r'\s+', ' ', text).strip()
    
    return text


# ==========================================
# 🚀 메인 실행 로직
# ==========================================
if not os.path.exists(INPUT_FILE):
    print(f"❌ Input file not found: {INPUT_FILE}")
    exit()

total_bytes = os.path.getsize(INPUT_FILE)
stats = {
    "total_read": 0,
    "spam_filtered": 0,
    "short_filtered": 0,
    "valid_saved": 0
}

print(f"🚀 Starting Final Cleaning on {INPUT_FILE}...")
print("   (Preserving valid long lines like novels, Removing 'Ook.' spam)")

with open(INPUT_FILE, 'r', encoding='utf-8') as f_in:
    with open(OUTPUT_FILE, 'w', encoding='utf-8') as f_out:
        
        with tqdm(total=total_bytes, unit='B', unit_scale=True, desc="Processing") as pbar:
            for line in f_in:
                pbar.update(len(line.encode('utf-8')))
                stats["total_read"] += 1
                
                # 빈 줄은 문서 구분자로 그대로 둠
                if not line.strip():
                    f_out.write('\n')
                    continue

                # 1단계: 내용 기반 스팸 필터링 (원본 텍스트로 검사)
                if is_spam_content(line):
                    stats["spam_filtered"] += 1
                    continue

                # 2단계: 정제 수행
                cleaned = clean_text_final(line)
                
                # 3단계: 정제 후 너무 짧아진 것(특수문자만 있던 줄 등) 제거
                if len(cleaned) < 10:
                    stats["short_filtered"] += 1
                    continue
                
                f_out.write(cleaned + '\n')
                stats["valid_saved"] += 1

print("\n" + "="*50)
print("✅ Cleaning Completed Successfully!")
print(f"📂 Output File: {os.path.abspath(OUTPUT_FILE)}")
print("-" * 50)
print(f"Total Lines Read    : {stats['total_read']:,}")
print(f"🗑️ Spam Filtered     : {stats['spam_filtered']:,} (Ook., Numbers, Garbㄹage)")
print(f"✂️ Short Filtered    : {stats['short_filtered']:,} (Too short after clean)")
print(f"📝 Valid Lines Saved : {stats['valid_saved']:,}")
print("="*50)

🚀 Starting Final Cleaning on cleaned/namuwiki_parsed.txt...
   (Preserving valid long lines like novels, Removing 'Ook.' spam)


Processing: 100%|██████████████████████████████████████████████████████████████████| 13.9M/13.9M [00:02<00:00, 5.53MB/s]


✅ Cleaning Completed Successfully!
📂 Output File: ./datasets/1.Wiki/cleaned/namuwiki_cleaned.txt
--------------------------------------------------
Total Lines Read    : 118,483
🗑️ Spam Filtered     : 28,641 (Ook., Numbers, Garbㄹage)
✂️ Short Filtered    : 3,206 (Too short after clean)
📝 Valid Lines Saved : 51,552


### 검증

In [6]:
TARGET_FILE_PATH = "cleaned/namuwiki_cleaned.txt"

# ==========================================
# 4. (수정됨) 잔여 특수문자 및 노이즈 분석
# ==========================================
def analyze_noise_characters(file_path, top_n=50):
    """
    결과 파일에서 한글, 영어, 숫자, 공백을 제외한 문자들의 빈도를 계산합니다.
    (Plain Text 전용)
    """
    print(f"\n[Analysis] Counting non-Korean/non-English/non-Numeric characters in {file_path}...")
    
    # 분석에서 '정상'으로 간주하고 제거할 패턴
    # 한글: 가-힣, ㄱ-ㅎ, ㅏ-ㅣ
    # 영어: a-zA-Z
    # 숫자: 0-9 (앞선 전처리에서 숫자를 살렸으므로 여기서도 정상 취급)
    # 공백: \s
    allowed_pattern = re.compile(r'[가-힣ㄱ-ㅎㅏ-ㅣa-zA-Z0-9\s]+')
    
    noise_counter = Counter()
    
    if not os.path.exists(file_path):
        print(f"❌ Error: File not found at {file_path}")
        return

    with open(file_path, 'r', encoding='utf-8') as f:
        # 파일 전체를 줄 단위로 읽으며 처리 (JSON 파싱 제거)
        # tqdm으로 진행률 표시
        lines = f.readlines() # 파일이 아주 크지 않다면 readlines로 읽어 진행률 표시가 정확함
        
        for line in tqdm(lines, desc="Analyzing Noise"):
            if not line: continue
            
            # 텍스트 가져오기 (이미 정제된 텍스트 파일이므로 line 자체가 텍스트)
            text = line
            
            # 정상 문자열(한글/영어/숫자/공백)을 빈 문자열로 치환하여 제거
            # 남은 것은 전처리에서 걸러지지 않은 특수문자들임
            noise_chars = allowed_pattern.sub('', text)
            
            # 카운터 업데이트 (남은 문자들만 셈)
            if noise_chars:
                noise_counter.update(noise_chars)

    # 결과 출력
    print(f"\n📊 Top {top_n} Other Characters (Potential Noise):")
    print("-" * 55)
    print(f"{'Rank':^5} | {'Char':^10} | {'Count':^12} | {'Hex Code':^10}")
    print("-" * 55)
    
    for rank, (char, count) in enumerate(noise_counter.most_common(top_n), 1):
        # 1. 출력 불가능하거나 공백인 문자는 눈에 보이게 표현 (repr 사용)
        char_display = repr(char) if not char.isprintable() or char.strip() == '' else char
        # 2. 유니코드 값을 16진수로 표현 (예: U+0041)
        hex_code = f"U+{ord(char):04X}"
        
        print(f"{rank:^5} | {char_display:^10} | {count:^12,} | {hex_code:^10}")

# ==========================================
# 실행
# ==========================================
if __name__ == "__main__":
    if os.path.exists(TARGET_FILE_PATH):
        analyze_noise_characters(TARGET_FILE_PATH)
    else:
        print(f"파일을 찾을 수 없습니다. 경로를 확인해주세요: {TARGET_FILE_PATH}")


[Analysis] Counting non-Korean/non-English/non-Numeric characters in cleaned/namuwiki_cleaned.txt...


Analyzing Noise: 100%|████████████████████████████████████████████████████████| 86636/86636 [00:00<00:00, 354852.05it/s]


📊 Top 50 Other Characters (Potential Noise):
-------------------------------------------------------
Rank  |    Char    |    Count     |  Hex Code 
-------------------------------------------------------
  1   |     .      |    90,809    |   U+002E  
  2   |     '      |    68,530    |   U+0027  
  3   |     ,      |    53,006    |   U+002C  
  4   |     )      |    27,527    |   U+0029  
  5   |     (      |    27,228    |   U+0028  
  6   |     -      |    15,312    |   U+002D  
  7   |     ~      |    7,010     |   U+007E  
  8   |     "      |    5,284     |   U+0022  
  9   |     %      |    4,052     |   U+0025  
 10   |     !      |    1,391     |   U+0021  
 11   |     ?      |     991      |   U+003F  


In [7]:
import os
REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CORPUS_DATA = os.path.join(REPO_ROOT, "data", "1.corpus")

# ==========================================
# 설정
# ==========================================
INPUT_FILE = os.path.join(CORPUS_DATA, "1.wiki/cleaned/namuwiki_cleaned.txt")  # 검증할 파일 경로
LONG_LINE_THRESHOLD = 10000           # 10,000자 이상이면 '괴물 문장'으로 간주

# 검증할 패턴 정의 (Regex)
patterns = {
    "1. URL/Links": re.compile(r'(https?://|www\.)[\w\-\./?=&%]+'),
    "2. Wiki Syntax": re.compile(r'(\[\[.*?\]\]|\{\{.*?\}\}|\|\|.*?\|\|)'),
    "3. HTML Entities": re.compile(r'&[a-zA-Z]{2,5};|&#[0-9]+;'),
}

# 통계 및 샘플 저장소
stats = {k: 0 for k in patterns.keys()}
stats["4. Long Lines"] = 0
samples = {k: [] for k in patterns.keys()}
samples["4. Long Lines"] = []

print(f"🔍 Inspecting {INPUT_FILE} for noise patterns...\n")

if not os.path.exists(INPUT_FILE):
    print("❌ 파일을 찾을 수 없습니다.")
    exit()

total_size = os.path.getsize(INPUT_FILE)

with open(INPUT_FILE, 'r', encoding='utf-8') as f:
    # 빠른 확인을 위해 전체를 다 돌지 않고, 문제가 발견될 때마다 로그를 찍도록 함
    with tqdm(total=total_size, unit='B', unit_scale=True, desc="Scanning") as pbar:
        for line_num, line in enumerate(f, 1):
            pbar.update(len(line.encode('utf-8')))
            
            line = line.strip()
            if not line: continue

            # -------------------------------------------------
            # 4. 괴물 문장 (Long Line) 검증
            # -------------------------------------------------
            if len(line) > LONG_LINE_THRESHOLD:
                stats["4. Long Lines"] += 1
                if len(samples["4. Long Lines"]) < 3: # 예시 3개만 저장
                    preview = line[:100] + " ... [생략] ... " + line[-100:]
                    samples["4. Long Lines"].append(f"(Line {line_num}, Length: {len(line):,}) {preview}")

            # -------------------------------------------------
            # 1, 2, 3번 패턴 검증
            # -------------------------------------------------
            for name, pattern in patterns.items():
                match = pattern.search(line)
                if match:
                    stats[name] += 1
                    if len(samples[name]) < 3: # 각 유형별 예시 3개만 저장
                        # 매칭된 부분만 잘라서 보여줌
                        detected_str = match.group(0)
                        # 너무 길면 자름
                        if len(detected_str) > 50: 
                            detected_str = detected_str[:50] + "..."
                        samples[name].append(f"(Line {line_num}) Found: '{detected_str}' in context: {line[:60]}...")

# ==========================================
# 결과 리포트 출력
# ==========================================
print("\n" + "="*60)
print(f"📊 NOISE VERIFICATION REPORT")
print("="*60)

for key in sorted(stats.keys()):
    count = stats[key]
    print(f"\n🔴 [{key}] Detected Lines: {count:,}")
    
    if count > 0:
        print(f"   --- Examples (First 3) ---")
        for i, example in enumerate(samples[key], 1):
            print(f"   {i}. {example}")
    else:
        print("   ✅ Clean (Not found)")

print("\n" + "="*60)

🔍 Inspecting ./cleaned/namuwiki_cleaned.txt for noise patterns...



Scanning: 100%|████████████████████████████████████████████████████████████████████| 11.9M/11.9M [00:00<00:00, 33.3MB/s]


📊 NOISE VERIFICATION REPORT

🔴 [1. URL/Links] Detected Lines: 0
   ✅ Clean (Not found)

🔴 [2. Wiki Syntax] Detected Lines: 0
   ✅ Clean (Not found)

🔴 [3. HTML Entities] Detected Lines: 0
   ✅ Clean (Not found)

🔴 [4. Long Lines] Detected Lines: 0
   ✅ Clean (Not found)

